# Household Energy Consumption - Data Cleaning & Preprocessing

This notebook handles loading the raw household energy consumption dataset, cleaning missing values, parsing timestamps, and resampling the minute-level data to hourly averages. Cleaning and resampling are critical preprocessing steps to prepare the data for exploratory analysis and modeling.

## Setup and Library Imports

We import `numpy` and `pandas` for core data manipulation and cleaning. We also disable warning messages to keep the notebook outputs clean.

In [58]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

## Load the Raw Dataset

We load the raw dataset which is semicolon-separated. Missing values represented as `?` are converted to `NaN`.

In [59]:
df = pd.read_csv(
    r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\household_power_consumption.txt",
    sep=";",
    na_values="?",
    low_memory=False
)

### Inspect Columns

We display the column names to verify their labels before parsing date and time columns.

In [60]:
df.columns

Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='str')

## Combine Date and Time

The raw dataset lists `Date` and `Time` separately as strings. We combine them into a single `Datetime` column and parse them into actual datetime objects. Setting `dayfirst=True` is necessary because the dates are in DD/MM/YYYY format.

In [61]:
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True
)

## Data Type Conversion

We ensure that `Global_active_power` is cast to numerical float type. Any problematic entries that cannot be parsed are converted to `NaN` using `errors="coerce"`.

In [62]:
df["Global_active_power"] = pd.to_numeric(
    df["Global_active_power"],
    errors="coerce"
)

### Inspect Pre-Cleaning Shape

We check the size of the dataset before dropping missing values to track how many records are removed.

In [63]:
df.shape

(2075259, 10)

## Drop Missing Values

We remove any rows that contain missing values (`NaN`). Dropping incomplete rows is necessary since downstream statistical and machine learning algorithms require complete data matrices.

In [64]:
df = df.dropna()

### Inspect Post-Cleaning Shape

We verify the dimensions of the dataset after removing missing rows.

In [65]:
df.shape

(2049280, 10)

## Set Datetime Index

We index the dataframe by the `Datetime` column. Having a datetime index is required for resampling and slicing operations in Pandas.

In [66]:
df = df.set_index("Datetime")

## Resampling to Hourly Averages

Minute-level data contains a high amount of high-frequency noise and is computationally expensive to model. We drop the redundant `Date` and `Time` columns and resample the remaining columns to hourly intervals, computing the mean value for each hour. This stabilizes patterns and reduces file size.

In [67]:
df.drop(columns=["Date", "Time"], inplace=True)

hourly = df.resample("h").mean()

### Preview Resampled Data

We display the top five rows of the hourly dataset to confirm the resampling was successful.

In [68]:
hourly.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667


## Save the Cleaned Hourly Dataset

Finally, we write the resampled hourly data to `data/household_power_consumption_cleaned.txt` in CSV format. Saving the preprocessed file enables other notebooks to load the clean data directly, separating data engineering from downstream modeling and analysis.

In [69]:
hourly.to_csv(r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\household_power_consumption_cleaned.txt")